In [ ]:
import tables
from ctapipe.visualization import CameraDisplay
from ctapipe.coordinates import EngineeringCameraFrame
from ctapipe_io_lst import LSTEventSource
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import  LinearRegression , RANSACRegressor
import statsmodels.api as sm
import time


In [ ]:
# CAMERA GEOMETRY
sa = LSTEventSource.create_subarray(tel_id=1)
focal_length = sa.tel[1].optics.equivalent_focal_length
camera_geom = sa.tel[1].camera.geometry

In [ ]:
# DL1 DATACHECK FILE
a = tables.open_file("/data/cta/users-ifae/moralejo/CTA/summer_student_2026/datacheck/datacheck_dl1_LST-1.Run24704.h5")

# We'll first focus on looking for clues at the 'cosmics' data. 
# flatfields and pedestals will be later checked
# one can take a look at the content at the cheatsheet

In [ ]:
# USEFUL VARIABLES
subruns = (a.root.dl1datacheck.cosmics.col('subrun_index')) #0,1,...,57 number of subruns in this run
n_subruns = len(subruns)
time = a.root.dl1datacheck.cosmics.col('elapsed_time')

# ROBUST FIT FUNCTION
# x: 1D array like subruns 58
# y: 2D array like cog_pixel 58 x 1855
# fit : 2D array 58 x 1855

def robust_fit(x, y):
    X = x.astype(float).reshape(-1, 1)
    fit = np.zeros_like(y, dtype=float)
    for p in range(y.shape[1]):
        yp = y[:, p]
        # First fit on all points.
        model = LinearRegression()
        model.fit(X, yp)
        # Keep only points that are not too far from the first fit.
        residual = np.abs(yp - model.predict(X))
        mad = np.median(np.abs(residual - np.median(residual)))
        cutoff = max(4.0 * mad, 3.0)
        good = residual <= cutoff
        # Refit using only the kept points.
        if good.sum() >= 2:
            model = LinearRegression()
            model.fit(X[good], yp[good])
            fit[:, p] = model.predict(X)
        else:
            fit[:, p] = np.median(yp)
    return fit


In [ ]:
def robust_fitt(x, y):  # SIN REFERENCIA DE Z THRESHOLD--------------------------
    X = x.astype(float).reshape(-1, 1)
    Xc = sm.add_constant(X)  # add column of ones -> [intercept, slope]

    fit = np.zeros_like(y, dtype=float)

    slope = np.zeros(y.shape[1])
    u_slope = np.zeros(y.shape[1])
    intercept = np.zeros(y.shape[1])
    u_intercept = np.zeros(y.shape[1])
    good_points = []  # list of (x_selected, y_selected) tuples, one per column

    for p in range(y.shape[1]):
        yp = y[:, p]

        model = sm.OLS(yp, Xc).fit()
        residual = np.abs(yp - model.predict(Xc))
        mad = np.median(np.abs(residual - np.median(residual)))
        cutoff = max(4.0 * mad, 3.0)
        good = residual <= cutoff

        if good.sum() >= 2:
            model = sm.OLS(yp[good], Xc[good]).fit()
            fit[:, p] = model.predict(Xc)
            intercept[p] = model.params[0]
            slope[p] = model.params[1]
            u_intercept[p] = model.bse[0]
            u_slope[p] = model.bse[1]
        else:
            fit[:, p] = np.median(yp)
            intercept[p] = np.median(yp)
            slope[p] = 0.0
            u_intercept[p] = np.nan
            u_slope[p] = np.nan

        good_points.append((x[good], yp[good]))

    return {
        "fit": fit,
        "slope": slope,
        "u_slope": u_slope,
        "intercept": intercept,
        "u_intercept": u_intercept,
        "good_points": good_points,
    }

In [ ]:
def robust_fitt(x, y, z_thresh=3.5):
    X = x.astype(float).reshape(-1, 1)
    Xc = sm.add_constant(X)
    fit = np.zeros_like(y, dtype=float)
    slope = np.zeros(y.shape[1])
    u_slope = np.zeros(y.shape[1])
    intercept = np.zeros(y.shape[1])
    u_intercept = np.zeros(y.shape[1])
    good_points = []

    for p in range(y.shape[1]):
        yp = y[:, p]
        model = sm.OLS(yp, Xc).fit()
        residual = np.abs(yp - model.predict(Xc))
        mad = np.median(np.abs(residual - np.median(residual)))
        mad = max(mad, 1e-12)  # avoid division by zero
        modified_z = 0.6745 * residual / mad
        good = modified_z <= z_thresh

        if good.sum() >= 2:
            model = sm.OLS(yp[good], Xc[good]).fit()
            fit[:, p] = model.predict(Xc)
            intercept[p] = model.params[0]
            slope[p] = model.params[1]
            u_intercept[p] = model.bse[0]
            u_slope[p] = model.bse[1]
        else:
            fit[:, p] = np.median(yp)
            intercept[p] = np.median(yp)
            slope[p] = 0.0
            u_intercept[p] = np.nan
            u_slope[p] = np.nan

        good_points.append((x[good], yp[good]))

    return {"fit": fit, "slope": slope, "u_slope": u_slope,
            "intercept": intercept, "u_intercept": u_intercept,
            "good_points": good_points}

In [ ]:
# FILTER 1: EVENT RATE
# we expect the event rate to present anomalies when a satellite comes in the FoV
# for a really bright one, the pixels in the camera might de-activate as a safety measure, making the rate drop
# for a sat not that bright, there might be a peak in the rate

num_events = a.root.dl1datacheck.cosmics.col('num_events') # events in each subrun
event_rate = num_events / time
event_rate_sigma = np.sqrt(np.maximum(num_events, 1.0)) / time  

# ROBUST FIT
event_rate_robfit = robust_fitt(subruns, event_rate.reshape(-1, 1))
event_rate_fit = event_rate_robfit["fit"].flatten()

# Selection criteria
erate_z_score = np.abs(event_rate - event_rate_fit) / event_rate_sigma
checks_1 = subruns[erate_z_score > 3]

print(checks_1, erate_z_score[checks_1])
    
print('Subruns to check =', checks_1)


plt.plot(subruns, event_rate)
plt.plot(subruns, event_rate_fit, label='Robust fit', color='red', linestyle='--')
plt.fill_between(
    subruns,
    event_rate_fit - 3 * event_rate_sigma,
    event_rate_fit + 3 * event_rate_sigma,
    color='lightgray', label='Fit ± 3σ'
)
plt.title('Event rate per subrun')
plt.ylabel('# events / s')
plt.xlabel('subrun index')
plt.legend()
plt.show()

# Fit parameters and number of selected points
slope, u_slope = event_rate_robfit["slope"][0], event_rate_robfit["u_slope"][0]
intercept, u_intercept = event_rate_robfit["intercept"][0], event_rate_robfit["u_intercept"][0]
n_good = len(event_rate_robfit["good_points"][0][1])  # length of the y-selected array
n_total = len(subruns)

print(f"slope     = {slope:.4f} ± {u_slope:.4f}")
print(f"intercept = {intercept:.4f} ± {u_intercept:.4f}")
print(f"good points = {n_good}/{n_total}")
#print(event_rate_robfit["good_points"][0][1])

In [ ]:
#%%time

# FILTER 2: EVENT RATE VS INTENSITY
# we expect the event rate per intensity distribuition to vary for an anomali with respect to the NSB

# Intensity = sum of charge over all the pixels
# Hist_intensity: for each subrun, contains the histogram of intensities
hist_intensity = a.root.dl1datacheck.cosmics.col('hist_intensity')

# Find the center of each bin for the histogram of intensity
intensity_bins = a.root.dl1datacheck.histogram_binning.col('hist_intensity')[0]
intensity = 0.5*(intensity_bins[1:]+intensity_bins[:-1])  

# Event rate per intensity value in each subrun----------------------------
for s in range(hist_intensity.shape[0]):  
    event_rate_i = hist_intensity[s] / time[s]
    plt.plot(intensity, event_rate_i)

plt.xlabel('Intensity')
plt.ylabel('Events/s')
plt.xscale('log')
plt.show()

# Note: in this case, we can see that for subrun 8 the part on the right has the shape of a particle shower, but seems to be suppressed due to the saturation of
# the pixels in the camera by the crossing of the satellite

# Therefore, to perform a more sensitive analysis we'll implement the Kolmogorov-Smirnov statistic, because we want to compare the whole shape of our distribuition
# with that end we implement the Kolmogorov-Smirnov parameter D = max |F(x) - Fn| which tells us how different are two cumulative distribuition functions
# we will compare the CDF for each subrun with its neighbours, bc the satellite is expected to crossthe FoV within a fine period of time

cdf = np.cumsum(hist_intensity / hist_intensity.sum(axis=1, keepdims=True), axis=1) #we need to normalize the histogram. 1 cdf per subrun
D = np.zeros(n_subruns) 

for s in range(0, n_subruns-1):
    D[s] = np.max(np.abs(cdf[s] - cdf[s+1])) #we compare the CDF in each subrun 's' with its neighbouring one 's+1'

#DO A ROBUST FIT OF D TO SELECT THE GOOD POINTS
D_robust_fit = robust_fitt(subruns, D.reshape(-1, 1))

# This is not a Poisson process therefore we cannot treat it as such
D_good = D_robust_fit["good_points"][0][1]  # y-selected values for this column
#print(f"good points = {len(D_good)}/{len(D)}")

# calculate the sigma for those good points so that it is not biased 
sigma_2 = np.std(D_robust_fit["fit"]-D_good)
deviations_2 = np.abs(D - np.median(D_good))/sigma_2


# PLOT
plt.errorbar(D_robust_fit["good_points"][0][0], D_robust_fit["good_points"][0][1], yerr=sigma_2, fmt='.', label='selected', markersize=3)
plt.plot(subruns, D, linewidth=1, color='lightblue', label='D')
plt.xlabel('Subrun')
plt.ylabel('D Kolmogorov')
plt.axhline(y= np.median(D_good), color='red', linestyle='--', linewidth=1, label='Median')
plt.legend()
plt.show()

# SUBRUNS TO CHECK
checks_2 = subruns[deviations_2 > 3]
print('Subruns to check =', checks_2)

for c in checks_2:
    print(c, deviations_2[c])

#stdev respecto a la mediana , ver cuales son compatibles con sigma=stdev

In [ ]:

for i in (0,1,8,23, 24, 74,75,140,141):
    camdisplay = CameraDisplay(camera_geom.transform_to(EngineeringCameraFrame()),
                          norm='lin', title=f'Rate of events with CoG in pixel subrun ¨{i}', 
                           image=(a.root.dl1datacheck.cosmics.col('cog_within_pixel')/ time[:, None])[i,:])
    camdisplay.autoupdate = True
    camdisplay.add_colorbar()
    plt.show()


#AGL: during the elapse time of the 9-th subrun, the number of cog within pixel is counted, this is what this plot shows

#AGL: we can see, for instance, that in the brightest pixels the CoG has fallen approx 1200 time within the subrun



In [ ]:
#%%time 
# FILTER 3: FLUCTUATIONS OF CoG WITHIN PIXEL
cog_pixel = a.root.dl1datacheck.cosmics.col('cog_within_pixel')  # shape: (n_subruns, n_pixels)

cog_rate = cog_pixel / time[:, None]
cog_sigma = np.sqrt(np.maximum(cog_pixel, 1.0)) / time[:, None] # Poisson-like uncertainty associated to each value


# ROBUST FIT TO SELECT PIXELS IN WHICH THE ANOMALY HAS FALLEN 
cog_fit = robust_fitt(subruns, cog_rate)["fit"]

# Selection criteria
cog_z_score = (cog_rate - cog_fit) / cog_sigma
anomaly_mask = (cog_z_score) > 3 # when taking np.abs(z_score), much more anomalous pixels appear

# SELECTION AND PLOT OF THE PIXELS
# Select subrun interesting to check according to this filter
checks_3 = subruns[np.argmax(anomaly_mask.sum(axis=1))]
print('Subruns to check =', checks_3)


anomaly_image = anomaly_mask[checks_3].astype(int)

# Plot of CoG rate within pixel for anomalous subrun
camdisplay = CameraDisplay(
    camera_geom.transform_to(EngineeringCameraFrame()),
    norm='lin',
    title=f'# of events with CoG in pixel in subrun {checks_3}',
    image=cog_rate[checks_3, :],
)
camdisplay.autoupdate = True
camdisplay.add_colorbar(label='CoG counts')
plt.show()

# Plot of anomalous pixels in anomalous subrun
camdisplay = CameraDisplay(
    camera_geom.transform_to(EngineeringCameraFrame()),
    title=f'Anomalous pixels in subrun {checks_3}',
    image=anomaly_image,
)
camdisplay.add_colorbar(label='Anomalous pixel (1) / normal pixel (0)')
plt.show()

# usar las incertidumbres en el ajuste

# output: .py con solo un run y parametros interesantes
# que explique de que filtro proviene de cada subrun
# resultados de los filtros para todos los subruns
# anotar coord. de los pixels anomalos


#sobre el py script python my_script.py datacheck... o pasarle un directorio
#import glob, glob,glob(path).
#.sort()   // nohup antes del comando y & (background)
# que de un archivo por run


# postponer el analisis incluyendo informacion por pixel en el siguiente nivel
# poodemos seguur usando z_score 
# esperamos que en el hist del z_score este centrado en 0, la cual cosa no pasa para este subrun tan brillante



In [ ]:
# FILTER 3: PATH OF COG - Sanity check with median method

# Plot the median value up to the anomalous subrun. We can see the median rate is at about

cog_pixel = a.root.dl1datacheck.cosmics.col('cog_within_pixel')  # shape: (n_subruns, n_pixels)

rate = cog_pixel / time[:, None] # rate of number of times the CoG of an event has fallen into that given pixel
sigma = np.sqrt(np.maximum(cog_pixel, 1.0)) / time[:, None]


# MEDIAN RATE FOR 'NORMAL' SUBRUNS
camdisplay = CameraDisplay(camera_geom.transform_to(EngineeringCameraFrame()),
                           norm='lin', title='Median rate of # of times the CoG of an event has fallen into each pixel bf anomaly', 
                           image=np.median(rate[:8,:], axis=0))
camdisplay.autoupdate = True
camdisplay.add_colorbar()
plt.show()


# RATE FOR THE ANOMALOUS SUBRUN
camdisplay = CameraDisplay(camera_geom.transform_to(EngineeringCameraFrame()),
                           norm='lin', title='Rate of # of times the CoG of an event has fallen into each pixel in anomaly', 
                           image=rate[8,:])
camdisplay.autoupdate = True
camdisplay.add_colorbar()
plt.show()


# RATE PER PIXEL ALONG SUBRUNS
plt.plot(subruns, rate[:, 800])
plt.title('Rate per subrun in one pixel')
#plt.show()


# ROBUST FIT TO SELECT PIXELS IN WHICH THE ANOMALY HAS FALLEN 

x = subruns.astype(float)
X = x.reshape(-1, 1)
residuals_list = []
fit = np.zeros_like(rate, dtype=float)

for p in range(rate.shape[1]):
    y = rate[:, p]

    # First fit on all points.
    model = LinearRegression()
    model.fit(X, y)

    # Keep only points that are not too far from the first fit.
    residual = np.abs(y - model.predict(X))
    mad = np.median(np.abs(residual - np.median(residual)))
    cutoff = max(4.0 * mad, 3.0)
    good = residual <= cutoff

    # Refit using only the kept points.
    if good.sum() >= 2:
        model = LinearRegression()
        model.fit(X[good], y[good])
        fit[:, p] = model.predict(X)
    else:
        fit[:, p] = np.median(y)

    
    residuals_list.append(y - fit[:, p])


# PLOT OF THE RESIDUALS PER PIXEL
#plt.plot(subruns, residuals_list[1200])
plt.title('Residuals per subrun in one pixel')
#plt.show()


# RATE ALONG SUBRUNS AND 3SIGMA

for pixel_id in [105]: # range(rate.shape[1]):
    plt.fill_between(subruns, (fit - 3*sigma)[:, pixel_id], (fit + 3*sigma)[:, pixel_id], color='lightblue', alpha=0.4)
    plt.errorbar(subruns, rate[:, pixel_id], yerr=sigma[:, pixel_id], fmt='.', markersize=2)
    plt.plot(subruns, fit[:, pixel_id], '-', color='red', label='linear fit')

        
plt.title('Rate of # of times the CoG of an event has fallen into that pixel per subrun')
plt.show()

In [ ]:
%%time 
# FILTER 3: FLUCTUATIONS OF CoG WITHIN PIXEL - OLD
cog_pixel = a.root.dl1datacheck.cosmics.col('cog_within_pixel')  # shape: (n_subruns, n_pixels)

rate = cog_pixel / time[:, None]
sigma = np.sqrt(np.maximum(cog_pixel, 1.0)) / time[:, None]


# ROBUST FIT TO SELECT PIXELS IN WHICH THE ANOMALY HAS FALLEN 
x = subruns.astype(float)
X = x.reshape(-1, 1)
residuals_list = []
fit = np.zeros_like(rate, dtype=float)

for p in range(rate.shape[1]):
    y = rate[:, p]

    # First fit on all points.
    model = LinearRegression()
    model.fit(X, y)

    # Keep only points that are not too far from the first fit.
    residual = np.abs(y - model.predict(X))
    mad = np.median(np.abs(residual - np.median(residual)))
    cutoff = max(4.0 * mad, 3.0)
    good = residual <= cutoff

    # Refit using only the kept points.
    if good.sum() >= 2:
        model = LinearRegression()
        model.fit(X[good], y[good])
        fit[:, p] = model.predict(X)
    else:
        fit[:, p] = np.median(y)

    residuals_list.append(y - fit[:, p])
        

z_score = (rate - fit) / sigma
anomaly_mask = z_score > 3 # when taking np.abs(z_score), much more anomalous pixels appear

# SELECTION AND PLOT OF THE PIXELS
s = 8
anomaly_image = anomaly_mask[s].astype(int)

camdisplay = CameraDisplay(
    camera_geom.transform_to(EngineeringCameraFrame()),
    title=f'Anomalous pixels in subrun {s}',
    image=anomaly_image,
)
camdisplay.add_colorbar(label='Anomalous pixel (1) / normal pixel (0)')
plt.show()

camdisplay = CameraDisplay(
    camera_geom.transform_to(EngineeringCameraFrame()),
    norm='lin',
    title=f'# of events with CoG in pixel in subrun {s}',
    image=rate[s, :],
)
camdisplay.autoupdate = True
camdisplay.add_colorbar(label='CoG counts')
plt.show()

# Select subrun interesting to check according to this filter
checks_3 = subruns[np.argmax(anomaly_mask.sum(axis=1))]
print('Subruns to check =', checks_3)

# output: .py con solo un run y parametros interesantes
# que explique de que filtro proviene de cada subrun
# resultados de los filtros para todos los subruns
# anotar coord. de los pixels anomalos

In [ ]:
camera_geom.pix_x[anomaly_mask[8]]

In [ ]:
%%time 
#RANSAC
# FILTER 3: PATH OF CoG

cog_pixel = a.root.dl1datacheck.cosmics.col('cog_within_pixel')  # shape: (n_subruns, n_pixels)
rate = cog_pixel / time[:, None]
sigma = np.sqrt(np.maximum(cog_pixel, 1.0)) / time[:, None]
# ROBUST FIT TO SELECT PIXELS IN WHICH THE ANOMALY HAS FALLEN 
x = subruns.astype(float)
X = x.reshape(-1, 1)
residuals_list = []
fit = np.zeros_like(rate, dtype=float)
for p in range(rate.shape[1]):
    y = rate[:, p]
    model = RANSACRegressor()
    model.fit(X, y)
    fit[:, p] = model.predict(X)
    residuals_list.append(y - fit[:, p])

z_score = (rate - fit) / sigma
anomaly_mask = np.abs(z_score) > 3 # when taking np.abs(z_score), much more anomalous pixels appear


for pixel_id in range(rate.shape[1]):
    plt.fill_between(subruns, (fit - 3*sigma)[:, pixel_id], (fit + 3*sigma)[:, pixel_id], color='lightgray', alpha=0.4)
    plt.errorbar(subruns, rate[:, pixel_id], yerr=sigma[:, pixel_id], fmt='.')
   # plt.plot(subruns, fit[:, pixel_id], '-', color='red', label='linear fit')

        
plt.title('Rate of # of times the CoG of an event has fallen into that pixel per subrun')
plt.show()



# SELECTION AND PLOT OF THE PIXELS
s = 8
anomaly_image = anomaly_mask[s].astype(int)

camdisplay = CameraDisplay(
    camera_geom.transform_to(EngineeringCameraFrame()),
    title=f'Anomalous pixels in subrun {s}',
    image=anomaly_image,
)
camdisplay.add_colorbar(label='Anomalous pixel (1) / normal pixel (0)')
plt.show()

camdisplay = CameraDisplay(
    camera_geom.transform_to(EngineeringCameraFrame()),
    norm='lin',
    title=f'# of events with CoG in pixel in subrun {s}',
    image=rate[s, :],
)
camdisplay.autoupdate = True
camdisplay.add_colorbar(label='CoG counts')
plt.show()

# Select subrun interesting to check according to this filter
checks_3 = subruns[np.argmax(anomaly_mask.sum(axis=1))]
print(checks_3)

# output: .py con solo un run y parametros interesantes
# que explique de que filtro proviene de cada subrun
# resultados de los filtros para todos los subruns
# anotar coord. de los pixels anomalos

In [ ]:
# FILTER: CHARGE FLUCTUATIONS

# basically check the fluctuations  bc we expect that when a continuous source of light such as the Moon or a satellite
# enters the FoV, the fluctuations in the charge are larger

charge_stdev = a.root.dl1datacheck.cosmics.col('charge_stddev') # Size: 58 subruns x 1855 pixels
charge_mean = a.root.dl1datacheck.cosmics.col('charge_mean')

# see if the stdev of the charge varies from the mean for each pixel
c_fluctuation = (charge_stdev )

# compare the fluctuation with its median, and take the maximum value of this quantity per subrun 
c_fluctuation_cleaned_max = (charge_stdev - np.median(charge_stdev)).max(axis=1)

# in this case it is better to plot it because we do not seek for a given subrun but for a more
# abrupt behaviour in the fluctuations in the case of the satellite 

plt.plot(subruns, charge_mean.max(axis=1))
plt.plot()
check_2 = subruns[ np.argmax(c_fluctuation_cleaned_max)]

print(check_2)